In [1]:
%pip uninstall fitz pymupdf -y

Found existing installation: PyMuPDF 1.27.1
Uninstalling PyMuPDF-1.27.1:
  Successfully uninstalled PyMuPDF-1.27.1
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install pymupdf

  Using cached pymupdf-1.27.1-cp310-abi3-win_amd64.whl.metadata (3.4 kB)
Using cached pymupdf-1.27.1-cp310-abi3-win_amd64.whl (19.2 MB)
Note: you may need to restart the kernel to use updated packages.


In [3]:
import fitz

In [4]:
%pip install ollama

Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install dotenv

Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install google-genai

Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
import re
import json
import fitz  # PyMuPDF
import ollama
from typing import List, Dict, Optional
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from google import genai

In [8]:
%pip install transformers 

Note: you may need to restart the kernel to use updated packages.


In [23]:
%pip install pydantic

Note: you may need to restart the kernel to use updated packages.


In [27]:
%pip install typing

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for typing: filename=typing-3.7.4.3-py3-none-any.whl size=26419 sha256=47f53e874c6559d636788047614926352ff2a6cff601f7c51e681c0b8e91ff02
  Stored in directory: c:\users\user\appdata\local\pip\cache\wheels\12\98\52\2bffe242a9a487f00886e43b8ed8dac46456702e11a0d6abef
Successfully built typing
Note: you may need to restart the kernel to use updated packages.


In [28]:
from pydantic import field_validator

In [29]:
# Load configuration
load_dotenv()

True

In [30]:
from typing import Any, Union, List, Dict, Optional
from pydantic import BaseModel, Field, field_validator

In [31]:
# --- 1. FLEXIBLE SCHEMA ---
# This schema is "bulletproof" - it won't crash if the AI sends data in the wrong format.
class DDRReport(BaseModel):
    property_issue_summary: Any 
    probable_root_cause: Any
    severity_assessment: Any
    recommended_actions: Any
    conflicts: Any
    missing_information: Any
    confidence_score: Any

    @field_validator('*', mode='after')
    @classmethod
    def force_string_conversion(cls, v):
        """Automatically converts lists/dicts from AI into readable strings."""
        if isinstance(v, list):
            # If it's a list of strings, join them. If it's a list of dicts, flatten them.
            items = []
            for i in v:
                if isinstance(i, dict):
                    items.append(", ".join([f"{k}: {val}" for k, val in i.items()]))
                else:
                    items.append(str(i))
            return ". ".join(items)
        if isinstance(v, dict):
            return ", ".join([f"{k}: {val}" for k, val in v.items()])
        return str(v)

In [32]:
# --- 2. THE CORE PIPELINE ---
class DDRPipeline:
    def __init__(self):
        self.model = os.getenv("OLLAMA_MODEL", "deepseek-r1:8b")

    def extract_text(self, pdf_path: str) -> str:
        """Extracts text from PDF robustly."""
        try:
            with fitz.open(pdf_path) as doc:
                return "".join([page.get_text() for page in doc])
        except Exception as e:
            print(f"Error reading PDF {pdf_path}: {e}")
            return ""

    def parse_thermal_data(self, text: str) -> List[ThermalReading]:
        """Regex-based extraction for thermal metrics."""
        pattern = r"Thermal image\s*:\s*(.*?)\s.*?Hotspot\s*:\s*(\d+\.?\d*)\s*°C.*?Coldspot\s*:\s*(\d+\.?\d*)"
        matches = re.findall(pattern, text, re.DOTALL)
        
        readings = []
        for m in matches:
            hot, cold = float(m[1]), float(m[2])
            delta = round(hot - cold, 2)
            readings.append(ThermalReading(
                image_id=m[0].strip(),
                hotspot=hot,
                coldspot=cold,
                delta=delta,
                anomaly_detected=delta > 4.0
            ))
        return readings

    def get_structured_report(self, inspection_text: str, thermal_readings: List[ThermalReading]) -> DDRReport:
        """Synthesizes data. Optimized to avoid Schema-Parroting error."""
        
        anomalies = [r.model_dump() for r in thermal_readings if r.anomaly_detected]
        
        # We use a simplified template instead of the full JSON schema to avoid confusion
        prompt = f"""
        Extract a Detailed Diagnostic Report (DDR) based on the data below.
        
        INSPECTION DATA:
        {inspection_text}
        
        THERMAL ANOMALIES:
        {json.dumps(anomalies)}
        
        RULES:
        1. SEVERITY: High, Moderate, or Low.
        2. CONFLICTS: Flag if text mentions leakage but thermal delta is low (<2.0).
        3. ROOT CAUSE: Be specific about plumbing vs seepage.

        Return ONLY a JSON object with these exact keys:
        "property_issue_summary", "probable_root_cause", "severity_assessment", 
        "recommended_actions", "conflicts", "missing_information", "confidence_score"
        """

        response = ollama.chat(
            model=self.model,
            messages=[{'role': 'user', 'content': prompt}],
            format="json", # This is key
            options={"temperature": 0.1}
        )
        
        # Extract content and remove DeepSeek's <tool_call> tags if they exist
        raw_content = response['message']['content']
        clean_json = re.sub(r'<tool_call>.*?<tool_call>', '', raw_content, flags=re.DOTALL).strip()
        
        return DDRReport.model_validate_json(clean_json)

In [33]:
# --- 3. RUNNER & FILE SAVER ---
if __name__ == "__main__":
    pipeline = DDRPipeline()

    print(f"--- DDR Pipeline Started ({pipeline.model}) ---")
    
    # 1. Load Documents
    insp_text = pipeline.extract_text("Sample_inputs/Sample Report.pdf")
    therm_text = pipeline.extract_text("Sample_inputs/Thermal Images.pdf")

    # 2. Process Data
    thermal_data = pipeline.parse_thermal_data(therm_text)

    # 3. Generate and Save
    try:
        # Use a safe slice of text to prevent memory/context overflow
        report = pipeline.get_structured_report(insp_text[:6000], thermal_data)
        
        # Save as JSON (Production Data)
        json_file = "final_ddr_report.json"
        with open(json_file, "w", encoding="utf-8") as f:
            f.write(report.model_dump_json(indent=4))
            
        # Save as Text (Human Readable)
        text_file = "DDR_Summary.txt"
        with open(text_file, "w", encoding="utf-8") as f:
            f.write(f"DDR SUMMARY REPORT\n{'='*20}\n")
            f.write(f"SEVERITY: {report.severity_assessment}\n")
            f.write(f"CAUSE: {report.probable_root_cause}\n\n")
            f.write(f"SUMMARY:\n{report.property_issue_summary}\n")
        
        print(f"\nSUCCESS!")
        print(f"Data saved to: {os.path.abspath(json_file)}")
        print(f"Read it here: {os.path.abspath(text_file)}")
        
    except Exception as e:
        print(f"Pipeline Error: {e}")

--- DDR Pipeline Started (deepseek-r1:8b) ---

SUCCESS!
Data saved to: d:\Projects\DDR\final_ddr_report.json
Read it here: d:\Projects\DDR\DDR_Summary.txt
